<a href="https://colab.research.google.com/github/davidlealo/sic_ia_2026/blob/main/clases/sic_2026_ia_Clase_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIC 2026 · Inteligencia Artificial
## Clase 20: avances del proyecto final

Jueves 19 de noviembre de 2026 · Sesión de trabajo en grupos

Esta clase fue tiempo protegido para avanzar en el proyecto final: revisión de equipos, dudas en la sala principal y trabajo en salas separadas. Este notebook reúne las respuestas a las preguntas que salieron, para que no dependan de haber estado en la sesión.

### Revisión de grupos


> Trabajo en salas

## 1. Criterios de la presentación final

- **Duración:** máximo **10 minutos**.
- **Qué mostrar:** el proyecto funcionando **y el código**. Las preguntas de evaluación van a estar centradas en ese código.
- **Modelos externos y herramientas de IA:** no hay límite; usen lo que sirva, idealmente gratuito (el programa no tiene presupuesto para servicios pagados). La condición es que **el equipo pueda explicar qué hace el código** y por qué tomó cada decisión.
- **Interfaz:** Gradio o Shiny no son obligatorios. Se mostraron porque son fáciles, pero pueden usar otra herramienta. Lo importante es resolver el problema con lo visto en el curso (machine learning, deep learning, PLN…).

## 2. ¿Un accuracy de 70% u 80% está mal?

No hay un umbral fijo: **la métrica se interpreta según el problema y los datos**.

- En imágenes o audio es normal tener métricas más bajas que con datos tabulares muy limpios.
- Una métrica **demasiado alta** también merece revisión: puede haber fuga de datos (información del test en el entrenamiento) o un conjunto de prueba demasiado fácil.
- En la presentación expliquen **contra qué comparan**: un modelo base (predecir siempre la clase más frecuente) o una versión anterior del modelo.

La celda siguiente calcula el modelo base para que tengan con qué comparar.

In [ ]:
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Reemplacen y_train / y_test por las etiquetas de su proyecto
y_train = np.array([0, 0, 0, 1, 1, 2, 0, 1, 0, 0])
y_test = np.array([0, 1, 0, 0, 2, 0])

base = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_train), 1)), y_train)
pred_base = base.predict(np.zeros((len(y_test), 1)))
print(f"Accuracy del modelo base (clase más frecuente): {accuracy_score(y_test, pred_base):.0%}")
print("Su modelo tiene que superar claramente este número para aportar algo.")

## 3. Que Colab no se lleve tu entrenamiento

Si Colab se desconecta en la época 25 de 50, no hay que empezar de cero. La idea que vimos en clase: **guardar en Google Drive, en cada época, los pesos y las métricas**, y al volver a correr el notebook retomar desde la última época guardada.

Keras ya trae las piezas:

- `BackupAndRestore`: guarda el estado del entrenamiento en cada época y, si lo vuelves a llamar, **retoma solo** desde donde quedó.
- `CSVLogger(append=True)`: deja las métricas de cada época en un CSV (lo pueden abrir en Google Sheets).
- `ModelCheckpoint`: guarda el mejor modelo según la métrica de validación.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
CARPETA = "/content/drive/MyDrive/proyecto_final/entrenamiento"  # cámbienla por la de su equipo
os.makedirs(CARPETA, exist_ok=True)

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Modelo y datos de ejemplo: reemplácenlos por los de su proyecto
(x_train, y_train), (x_val, y_val) = keras.datasets.mnist.load_data()
x_train, x_val = x_train / 255.0, x_val / 255.0

modelo = keras.Sequential([
    keras.layers.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    keras.callbacks.BackupAndRestore(backup_dir=f"{CARPETA}/respaldo"),
    keras.callbacks.CSVLogger(f"{CARPETA}/metricas.csv", append=True),
    keras.callbacks.ModelCheckpoint(f"{CARPETA}/mejor_modelo.keras",
                                    monitor="val_accuracy", save_best_only=True),
]

# Si Colab se corta, vuelvan a correr todo el notebook: el entrenamiento sigue desde la última época.
historia = modelo.fit(x_train, y_train, validation_data=(x_val, y_val),
                      epochs=20, callbacks=callbacks)

In [ ]:
import pandas as pd

# Métricas de todas las épocas, incluidas las de antes de la desconexión
pd.read_csv(f"{CARPETA}/metricas.csv").tail()

## 4. ¿Pocos datos reales? Datos sintéticos

Si tienen pocas muestras (por ejemplo, pocas grabaciones de un sonido), **generar datos sintéticos es una estrategia válida** y se usa en la industria: la familia de modelos **Phi-3 de Microsoft** se entrenó con una gran cantidad de datos sintéticos combinados con datos públicos filtrados.

Cómo usarlos con cuidado:

1. Entrenen una primera versión con lo que tienen.
2. Agreguen datos sintéticos y vuelvan a entrenar.
3. **Evalúen siempre con datos reales**, nunca con sintéticos: es la única forma de saber si ayudaron.
4. Miren los errores, ajusten y repitan.

En la **Clase 21** profundizamos esto con un caso: datos sintéticos para clasificar cantos de aves chilenas.

## Dudas

Las dudas que queden pendientes, por Discord.